# The t-Test

**DS4DH Practice Pack · Module 04 — Statistical Inference**

*Technique:* One-sample t-test on paired within-CSD differences

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/ds4dh-colab-pack/blob/main/notebooks/04a_t_test.ipynb)

Data: `merged_dataset.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
from scipy import stats

# This notebook reads the CSVs sitting next to it. In Colab you will be asked
# to upload them from the pack's data/ folder.
NEEDED = ['merged_dataset.csv']

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df       = pd.read_csv('merged_dataset.csv')
CITIES = ['Montréal', 'Toronto', 'Edmonton', 'Vancouver']

print(f'Loaded. df has {len(df):,} rows and {df.shape[1]} columns.')

## What this notebook does

Module 02 produced a gap. This notebook asks the only question that can be asked
of a gap computed from a sample: **could a gap this large have arisen if the true
difference were zero?**

That is all a t-test answers. It does not say the gap is important, or real in
any deeper sense, or caused by anything.

## Why *one*-sample, not two

The instinct is a two-sample test: immigrant renters vs non-immigrant renters.
That would be wrong here.

The two groups are not independent samples — they are **the same municipalities,
measured twice**. Pairing within a CSD removes the place from the comparison
entirely. So the right object is the vector of within-CSD differences, and the
right test is whether *its mean* differs from zero.

In [ ]:
csd = df.dropna(subset=['csd_code'])

imm = csd[csd['immigrant_status'] == 'Immigrant'][
    ['csd_code', 'geography_name', 'cma', 'Renter']].copy()
imm.columns = ['csd_code', 'geography_name', 'cma', 'renter_stir_imm']

nim = csd[csd['immigrant_status'] == 'Non-immigrants'][['csd_code', 'Renter']].copy()
nim.columns = ['csd_code', 'renter_stir_nim']

penalty_df = imm.merge(nim, on='csd_code', how='inner')
penalty_df['penalty'] = penalty_df['renter_stir_imm'] - penalty_df['renter_stir_nim']
penalty_df = penalty_df.dropna(subset=['penalty'])
penalty_df = penalty_df[penalty_df['cma'].isin(CITIES)]

print(f'{len(penalty_df)} CSDs where BOTH groups have a reported renter STIR')
print()
print(penalty_df[['geography_name', 'cma', 'renter_stir_imm',
                  'renter_stir_nim', 'penalty']].head(8).to_string(index=False))

In [ ]:
# The test, one city at a time.
print(f'{"City":<12}{"n":>5}{"mean gap":>11}{"t":>9}{"p":>11}')
print('-' * 48)
results = {}
for city in CITIES:
    s = penalty_df[penalty_df['cma'] == city]['penalty']
    t, p = stats.ttest_1samp(s, 0)
    results[city] = {'n': len(s), 'mean': s.mean(), 't': t, 'p': p}
    print(f'{city:<12}{len(s):>5}{s.mean():>+11.2f}{t:>+9.3f}{p:>11.5f}')

## Reading this table

At the conventional α = 0.05, one city is below the threshold and three are
nowhere near it. Note especially:

- three cities have **positive** mean gaps with p-values between 0.36 and 0.66 —
  these are indistinguishable from noise
- the one city that does cross has a **negative** gap

The direction matters. The significant finding runs opposite to the pattern the
course has been building toward since Module 02.

In [ ]:
ALPHA = 0.05
for city, r in results.items():
    verdict = 'reject H0' if r['p'] < ALPHA else 'cannot reject H0'
    print(f'{city:<12} p={r["p"]:.5f}  {verdict}')
print()
print('H0 here is "the mean within-CSD gap is zero".')
print('"Cannot reject" is not "there is no gap" — it is "this sample cannot tell".')

### 🔧 Your turn 1

Change `ALPHA` to `0.01` and re-run.

Does the set of cities that reject H0 change? A conclusion that flips between
0.05 and 0.01 is a borderline conclusion and should be reported as such.

## What the test assumes

A t-test assumes the differences are roughly normally distributed, or that the
sample is large enough for the central limit theorem to carry it. With n = 13 in
one city, that is worth checking rather than assuming.

In [ ]:
print(f'{"City":<12}{"n":>5}{"skew":>9}{"shapiro p":>12}{"normality":>14}')
print('-' * 52)
for city in CITIES:
    s = penalty_df[penalty_df['cma'] == city]['penalty']
    sk = stats.skew(s)
    w, sp = stats.shapiro(s)
    flag = 'ok' if sp > 0.05 else 'questionable'
    print(f'{city:<12}{len(s):>5}{sk:>9.2f}{sp:>12.4f}{flag:>14}')
print()
print('Where normality is questionable, a rank-based test is the safer check.')

In [ ]:
# Wilcoxon signed-rank: same question, no normality assumption.
print(f'{"City":<12}{"t-test p":>11}{"wilcoxon p":>13}{"agree?":>9}')
print('-' * 46)
for city in CITIES:
    s = penalty_df[penalty_df['cma'] == city]['penalty']
    _, tp = stats.ttest_1samp(s, 0)
    try:
        _, wp = stats.wilcoxon(s)
    except ValueError:
        wp = float('nan')
    agree = (tp < 0.05) == (wp < 0.05)
    print(f'{city:<12}{tp:>11.5f}{wp:>13.5f}{("yes" if agree else "NO"):>9}')

### 🔧 Your turn 2

Do the two tests agree on every city?

When a parametric and a non-parametric test disagree, which do you report — and
what does the disagreement itself tell a reader?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

**Your turn 1.** Edmonton (p ≈ 0.0057) still rejects at α = 0.01; the other three
are unaffected because their p-values are far above both thresholds. Nothing here
is borderline, which is a comfortable position to be in — it means the conclusion
does not depend on a threshold convention.

**Your turn 2.** The tests agree on all four cities. When they do not, the usual
report is the non-parametric result, with the disagreement itself stated: it tells
the reader the conclusion depends on a distributional assumption rather than on
the data alone. Silently picking whichever gives the smaller p-value is the
canonical form of p-hacking.

One further point: running four tests and reporting the one that crossed 0.05 is
itself a problem, regardless of which test you used. That is the subject of the
next notebook.

</details>

## Where this stops

You have four p-values. You do not yet have a defensible conclusion, because you
asked four questions and are about to report the one that answered the way you
hoped. Notebook 04b fixes that.